# UZHAVAN M4 v1: EfficientNetV2-S training on Colab GPU

**What this does:** trains the approved M4 disease model (6 crops, leakage-safe split `m4_v1_split_seed20260925`) for seeds 1, 2 and 3.
**What it does NOT do:** evaluate on test or cross-source data. That happens independently on the Mac (`eval_m4.py`).

**Before running:**
1. Upload `m4_v1_colab.zip` (≈1.1 GB) to Google Drive at `MyDrive/uzhavan_m4/`.
2. Runtime → Change runtime type → **GPU** (T4 or better).
3. Run the cells top to bottom. You can re-run the seed cell after a disconnect: finished seeds are skipped, and unfinished run folders are renamed `*_INCOMPLETE_<time>`, never deleted or overwritten.

Data: 5 CC BY 4.0 sources (see `ATTRIBUTION.md` inside the zip). Do not share the zip beyond this training use without that attribution.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/uzhavan_m4'
RUNS = f'{DRIVE}/runs'
import os; os.makedirs(RUNS, exist_ok=True)
print(os.listdir(DRIVE))

In [ ]:
# GPU check: stop if there is no CUDA
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

In [ ]:
# Copy the package to local disk (much faster than reading from Drive) and verify its integrity
import hashlib, json, shutil, zipfile, time
t=time.time()
if not os.path.exists('/content/pkg/manifest.csv'):
    shutil.copy(f'{DRIVE}/m4_v1_colab.zip', '/content/m4_v1_colab.zip')
    zipfile.ZipFile('/content/m4_v1_colab.zip').extractall('/content/pkg')
meta = json.load(open('/content/pkg/package.json'))
sha = hashlib.sha256(open('/content/pkg/manifest.csv','rb').read()).hexdigest()
assert sha == meta['manifest_sha256'], 'manifest checksum mismatch: re-upload the zip'
n = len(os.listdir('/content/pkg/images')); assert n == meta['images'], f'{n} images, expected {meta["images"]}'
print('package OK:', n, 'images, manifest', sha[:12], f'{time.time()-t:.0f}s')

In [ ]:
# 1) GPU smoke test (2 batches). Writes to local /content only, never to Drive runs
!rm -rf /content/smoke && python /content/pkg/train_m4.py --pkg /content/pkg --out /content/smoke --seed 1 --epochs 1 --bs 32 --workers 2 --smoke

In [ ]:
# 2) Three-seed experiment. Re-run this cell after any disconnect: finished seeds are skipped.
import glob, json, subprocess, datetime
def finished(seed):
    for f in glob.glob(f'{RUNS}/UZ-M4-v1.0-*-s{seed}/experiment.json'):
        if json.load(open(f)).get('status') == 'experiment': return f
for d in glob.glob(f'{RUNS}/UZ-M4-v1.0-*'):
    if 'INCOMPLETE' not in d and not os.path.exists(f'{d}/experiment.json'):
        new = f"{d}_INCOMPLETE_{datetime.datetime.now():%Y%m%d%H%M}"; os.rename(d, new); print('preserved partial run ->', new)
for seed in (1, 2, 3):
    done = finished(seed)
    if done: print(f'seed {seed} already finished: {done}'); continue
    print(f'=== seed {seed} ===', flush=True)
    log = f'{RUNS}/train_seed{seed}_{datetime.datetime.now():%Y%m%d%H%M}.log'
    with open(log, 'w') as fh:
        p = subprocess.Popen(['python','/content/pkg/train_m4.py','--pkg','/content/pkg','--out',RUNS,'--seed',str(seed),'--epochs','30','--bs','32','--workers','2'],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout: print(line, end=''); fh.write(line); fh.flush()
    assert p.wait() == 0, f'seed {seed} failed: see {log}'

In [ ]:
# 3) Summary: download the whole MyDrive/uzhavan_m4/runs folder to the Mac for independent evaluation
for f in sorted(glob.glob(f'{RUNS}/UZ-M4-v1.0-*/experiment.json')):
    e = json.load(open(f)); print(e['experiment_id'], e['status'], 'best val mean macro-F1', round(e['best_val_mean_macro_f1'],4), 'time', round(e['training_duration_s']/3600,2),'h', e['hardware'])